# CDC-3 — Snapshot vs streaming phases

**Break → Detect → Fix → Prove.** On its first start a Debezium connector runs in **two phases**:
an **initial snapshot** (it reads every existing row and emits one **`r`** event per row — a
consistent picture of the table *as it already is*), then **streaming** (it tails the WAL and emits
**`c`/`u`/`d`** for changes that happen *from then on*). The connector's **`snapshot.mode`** decides
whether that first phase runs. Get it wrong and you either re-read a huge table you didn't need, or
**silently miss every row that already existed**.

We prove the two observable, deterministic outcomes on **two separate tables**:
- `snapshot.mode="initial"` → **`r`=N** (snapshot) then a post-start insert shows a **`c`** (streaming).
- `snapshot.mode="never"`   → **no `r` at all**; a post-start insert shows exactly **one `c`**.

**Prerequisite:** `make cdc-up`. **Laptop-safe:** two ≤20-row tables, bounded topic reads, both
connectors/tables torn down at start and end.

In [ ]:
from common import cdc_helpers as cdc
import json, time

# Two isolated demos so they never interfere: one per snapshot.mode.
INIT_CONN,  INIT_TABLE  = "cdc3-orders",       "cdc3_orders"        # snapshot.mode=initial
NEVER_CONN, NEVER_TABLE = "cdc3-orders-never",  "cdc3_orders_never"  # snapshot.mode=never
N = 20

print("Connect REST up:", cdc.connect_up(timeout=40))

# Clean slate (idempotent re-run): drop both connectors + slots + tables + topics.
cdc.teardown(INIT_CONN,  INIT_TABLE)
cdc.teardown(NEVER_CONN, NEVER_TABLE)
print("torn down any previous CDC-3 state")

## A note on what we can trigger (the honesty rule)

Two things here are deterministic and we prove them outright: **`initial`** emits `r`=N then `c`, and
**`never`** emits `c` only. The third — *interrupting a snapshot mid-way* — we **describe** rather than
stage: our table is so tiny the snapshot finishes in well under a second, so there's no window to crash
inside (section 5). This is the same stance as the SPK-2/SPK-3 OOM modules. The notebook still runs
top-to-bottom.

## Part A — `snapshot.mode="initial"`: snapshot **then** stream

Seed N rows *first*, then bring up the connector. With `initial`, the connector reads those N rows as a
consistent snapshot (one **`r`** event each) before it starts streaming the WAL.

In [ ]:
cdc.seed_orders(INIT_TABLE, n=N)                 # N pre-existing rows BEFORE the connector starts
INIT_TOPIC = cdc.topic_name(INIT_TABLE)

cfg = cdc.debezium_pg_config(INIT_CONN, INIT_TABLE, snapshot_mode="initial")
print("snapshot.mode =", cfg["snapshot.mode"], "| topic =", INIT_TOPIC)

reg = cdc.register_connector(INIT_CONN, cfg)
print("register -> HTTP", reg["status"])          # 201
print("state    ->", cdc.wait_for_connector(INIT_CONN, timeout=60))  # RUNNING

In [ ]:
# Snapshot phase: with no changes yet, every event is a READ (op="r"), one per existing row.
init_snapshot = cdc.read_cdc_events(INIT_TOPIC)
print("op counts (snapshot only):", cdc.op_counts(init_snapshot))   # {'r': 20}

r0 = [e for e in init_snapshot if e["op"] == "r"][0]
print("one snapshot event -> before:", r0["before"], "| after:", r0["after"], "| op:", r0["op"])

### Now a live change → a `c` (streaming) on the **same** topic

The connector is past its snapshot and streaming the WAL. An `INSERT` made *after* `RUNNING` is decoded
into a **create (`c`)** event and lands on the same topic as the snapshot `r` events.

In [ ]:
cdc.pg_exec(f"INSERT INTO public.{INIT_TABLE}(id,customer,amount,status) "
            "VALUES (100,'live',9.90,'NEW')")
time.sleep(5)                                      # streaming has a small WAL-decode lag

init_after = cdc.read_cdc_events(INIT_TOPIC)
init_counts = cdc.op_counts(init_after)
print("op counts (snapshot + streaming):", init_counts)   # {'r': 20, 'c': 1}

c0 = [e for e in init_after if e["op"] == "c"][0]
print("create event after:", c0["after"])

## Part B — `snapshot.mode="never"`: stream only, **no snapshot**

A *separate* fresh table, seeded with the same N rows, captured by a *second* connector with
`snapshot.mode="never"`. The connector skips the snapshot entirely: those N pre-existing rows are
**never emitted**. Only changes made *after* it starts will appear.

In [ ]:
cdc.seed_orders(NEVER_TABLE, n=N)                # N rows that EXIST before the connector — and will be skipped
NEVER_TOPIC = cdc.topic_name(NEVER_TABLE)

cfg_never = cdc.debezium_pg_config(NEVER_CONN, NEVER_TABLE, snapshot_mode="never")
print("snapshot.mode =", cfg_never["snapshot.mode"], "| topic =", NEVER_TOPIC)

reg = cdc.register_connector(NEVER_CONN, cfg_never)
print("register -> HTTP", reg["status"])           # 201
print("state    ->", cdc.wait_for_connector(NEVER_CONN, timeout=60))  # RUNNING

In [ ]:
# No snapshot ran: the N pre-existing rows produce NOTHING. The topic is empty (or not yet created).
never_before = cdc.read_cdc_events(NEVER_TOPIC)
print("op counts (never, before any change):", cdc.op_counts(never_before))   # {}  -> zero r
print("events seen:", len(never_before))

In [ ]:
# A change made AFTER the connector started is the only thing it sees.
cdc.pg_exec(f"INSERT INTO public.{NEVER_TABLE}(id,customer,amount,status) "
            "VALUES (200,'post-start',5.50,'NEW')")
time.sleep(5)

never_after = cdc.read_cdc_events(NEVER_TOPIC)
never_counts = cdc.op_counts(never_after)
print("op counts (never, after one post-start insert):", never_counts)   # {'c': 1}  -> still zero r

## Detect & Prove — the two modes side by side

The **`r` count is the entire difference**: `initial` carries the N snapshot rows, `never` carries
none. Both connectors are healthy (`RUNNING`) — this is a *configuration* difference, not a failure.
The N missing rows under `never` are exactly the history a "we only care about changes from now"
misconfiguration loses, **silently**.

In [ ]:
print("snapshot.mode contrast (op_counts):")
print(f"  initial ({INIT_TABLE:<18}) -> {init_counts}")    # {'r': 20, 'c': 1}
print(f"  never   ({NEVER_TABLE:<18}) -> {never_counts}")  # {'c': 1}

r_initial = init_counts.get("r", 0)
r_never   = never_counts.get("r", 0)
print(f"\nsnapshot (r) events:  initial={r_initial}   never={r_never}   "
      f"(difference = {r_initial - r_never} rows the 'never' pipeline never saw)")

for name in (INIT_CONN, NEVER_CONN):
    st = cdc.connector_status(name)
    print(f"  {name:<20} connector={st['connector']['state']} "
          f"tasks={[t['state'] for t in st['tasks']]}")

## Diagnose, the `snapshot.mode` matrix & interrupting a snapshot

**Two phases, one slot, one topic.** On first start the connector snapshots the table at a single WAL
position (every row `op="r"`), then streams from exactly that LSN forward (`c`/`u`/`d`). Both phases
publish to the same `<prefix>.<schema>.<table>` topic, so a phase-agnostic consumer gets a seamless
full picture. `snapshot.mode` only decides whether the first phase happens.

| Mode | First start | On restart (offsets exist) | Use when |
|------|-------------|----------------------------|----------|
| `initial` *(default)* | snapshot **then** stream | stream only (no re-snapshot) | full history **and** ongoing changes |
| `initial_only` | snapshot **then stop** | does nothing | a **one-shot backfill** of the current table |
| `never` | stream only — **no** snapshot | stream only | history is irrelevant; changes from now on |
| `when_needed` | snapshot only if it can't resume | snapshot if offsets are gone, else stream | self-healing after WAL/offset loss |
| `no_data` / `schema_only` | capture **schema**, no rows, then stream | stream | need the schema to decode but not existing rows |

**Interrupting a snapshot (described, not triggered — the honesty rule).** The default snapshot is
**not incremental**: it has **no per-row checkpoint**. If the connector crashes part-way through the
initial read, on restart it has no committed snapshot offset to resume from, so it **restarts the
snapshot from scratch**, re-reading every row:

```python
# Conceptually, a crash during the initial snapshot of a huge table:
#   register_connector(..., snapshot_mode="initial")   # begins reading 1e9 rows as 'r' events
#   <connector pod is killed at row 400M>              # NO per-row checkpoint was committed
#   <restart>  -> snapshot starts again at row 0        # the 400M already read are re-emitted
```

Our table snapshots in well under a second, so there is no deterministic window to crash inside — hence
we describe it. The fix is **incremental snapshots**: Debezium reads the table in **chunks** driven by a
**signal table** (an `execute-snapshot` signal), committing progress per chunk, so a restart **resumes**
mid-table and runs **concurrently** with streaming instead of blocking it:

```python
# Trigger an ad-hoc incremental snapshot by writing to the configured signal table:
#   INSERT INTO public.dbz_signal (id, type, data)
#   VALUES ('adhoc-1', 'execute-snapshot',
#           '{"data-collections": ["public.cdc3_orders"]}');
# (requires signal.data.collection + the incremental snapshot feature on the connector)
```

## Fix / guidance

- **Pick `snapshot.mode` for your restart & backfill needs, not by default.** Full mirror → `initial`;
  one-time copy → `initial_only`; go-forward-only → `never` (only if you truly don't need existing rows).
- **Large tables → incremental snapshots** (signal table + `execute-snapshot`): chunked, **resumable**,
  concurrent with streaming — no "crash mid-snapshot ⇒ start over" cliff.
- **`initial_only` for backfills** — it snapshots and **stops**, leaving no idle slot pinning WAL (CDC-5).
- **Make the sink phase-agnostic** — treat `r` and `c`/`u`/`d` as the same idempotent upsert (CDC-7) so
  the snapshot→streaming transition is invisible downstream.

## Teardown

Delete **both** connectors, drop their replication slots, drop both tables, delete both topics.

In [ ]:
cdc.teardown(INIT_CONN,  INIT_TABLE)
cdc.teardown(NEVER_CONN, NEVER_TABLE)
print("slots now:", cdc.list_slots(), "| make clean clears any remaining .tmp/ state")